# Livrable 3 - Groupe 1

## Contenu du livrable

Le but est de générer les légendes correspondant aux photos débruitées précédemment. Nous nous appuierons sur les CNN et sur les RNN pour traiter nos photos et générer les légendes. 

## Chargement des bibliothèques

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np
import os
import time
import json
import pickle
import random
import re
import collections
from glob import glob
from PIL import Image
from tqdm import tqdm
from typing import Dict, List, Tuple, Any

In [ ]:
# Configuration générale
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

## Chargement des données & définition des constantes

Cette classe Config a trois rôles principaux :

Définir les chemins d’accès aux données :
Elle indique où sont stockées les images d'entraînement, les annotations COCO, les checkpoints, etc.

Fixer les paramètres de traitement et d’apprentissage :
Taille des images, batch size, taux de validation, taille du vocabulaire, etc. Ces valeurs influencent directement la performance et la capacité du modèle.

Spécifier l’architecture du modèle :
Dimensions des embeddings de mots, nombre d’unités dans les couches RNN, et dimensions des features extraites par le CNN.



In [ ]:
class Config:
    """Classe de configuration centralisée pour tous les paramètres du modèle"""
    # Chemins des données
    DATA_PATH = os.path.abspath('./COCO')
    ANNOTATION_FILE = os.path.join(DATA_PATH, "annotations/captions_train2014.json")
    IMAGE_DIR = os.path.join(DATA_PATH, "train2014/")
    CHECKPOINT_PATH = "./checkpoints/train"
    MODEL_SAVE_PATH = "./saved_models"
    
    # Paramètres des données
    VALIDATION_SPLIT = 0.2
    IMG_SIZE = (299, 299)  # Taille standard pour InceptionV3
    BUFFER_SIZE = 1000
    BATCH_SIZE = 32
    
    # Paramètres du modèle
    EMBEDDING_DIM = 256
    UNITS = 512
    VOCAB_SIZE = 5000  # Nombre de mots à conserver dans le vocabulaire
    FEATURES_SHAPE = 2048
    ATTENTION_FEATURES_SHAPE = 64
    
    # Paramètres d'entraînement
    EPOCHS = 20
    LEARNING_RATE = 1e-3

In [ ]:
def setup_directories() -> None:
    """Crée les répertoires nécessaires s'ils n'existent pas"""
    os.makedirs(Config.CHECKPOINT_PATH, exist_ok=True)
    os.makedirs(Config.MODEL_SAVE_PATH, exist_ok=True)

### Pré-traitement des annotations

Cette étape vise à préparer les légendes du dataset COCO pour l’entraînement. Les descriptions textuelles sont extraites, nettoyées, et associées à leurs images respectives. Chaque légende est encadrée par des balises de début et de fin pour guider le modèle lors de la génération.

Le résultat est une structure claire contenant toutes les paires image-légende, prête à être utilisée dans le pipeline de captioning. C’est une étape clé pour garantir la cohérence des données et permettre un apprentissage efficace.

In [ ]:
def load_annotations() -> Tuple[List[str], List[str], Dict]:
    """
    Charge et prétraite les annotations COCO
    
    Returns:
        Tuple contenant:
        - Liste des légendes
        - Liste des chemins d'images correspondants
        - Dictionnaire associant les chemins d'images à leurs légendes
    """
    print("Chargement des annotations...")
    with open(Config.ANNOTATION_FILE, 'r') as f:
        annotations = json.load(f)

    # Grouper toutes les annotations par identifiant d'image
    image_path_to_caption = collections.defaultdict(list)
    for val in annotations['annotations']:
        # Marquer le début et la fin de chaque légende
        caption = '<start> ' + val['caption'] + ' <end>'
        # L'identifiant d'une image fait partie de son chemin d'accès
        image_path = os.path.join(Config.IMAGE_DIR, f"COCO_train2014_{val['image_id']:012d}.jpg")
        # Ajout de la légende associée à l'image
        image_path_to_caption[image_path].append(caption)

    image_paths = list(image_path_to_caption.keys())

    # Création des listes finales
    captions = []  # Toutes les légendes
    img_name_vector = []  # Chemins d'images correspondants
    
    for img_path in image_paths:
        captions.extend(image_path_to_caption[img_path])
        img_name_vector.extend([img_path] * len(image_path_to_caption[img_path]))
    
    return captions, img_name_vector, image_path_to_caption

### Pré-traitement des images

Cette fonction permet de charger une image depuis son chemin, de la redimensionner à la taille attendue par le modèle InceptionV3, et de l’adapter au format d’entrée du réseau (normalisation, etc.).

Elle prépare ainsi chaque image dans un format exploitable directement par le modèle pour l’extraction de caractéristiques visuelles.

In [ ]:
def load_and_preprocess_image(image_path: str) -> Tuple[tf.Tensor, str]:
    """
    Charge et prétraite une image pour l'entrée du modèle InceptionV3
    
    Args:
        image_path: Chemin vers l'image à charger
        
    Returns:
        Tuple contenant l'image prétraitée et son chemin
    """
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, Config.IMG_SIZE)
    image = tf.keras.applications.inception_v3.preprocess_input(image)
    return image, image_path

### Tokenisation des légendes

On transforme les légendes textuelles en séquences numériques pour l’entraînement du modèle. Un tokenizer est d’abord construit à partir de l’ensemble des légendes, en conservant un vocabulaire limité (défini dans la configuration). Chaque mot est alors remplacé par un entier correspondant à son index dans ce vocabulaire.

Les séquences obtenues sont ensuite normalisées en longueur grâce au padding, ce qui permet de les traiter en lot (batch). On obtient au final un tableau de vecteurs numériques, la longueur maximale des légendes, ainsi que l’objet tokenizer qui sera utilisé plus tard pour la génération de texte.

In [ ]:
def build_tokenizer(captions: List[str]) -> Tuple[np.ndarray, int, Any]:
    """
    Crée un tokenizer pour les légendes et vectorise le texte
    
    Args:
        captions: Liste des légendes à tokenizer
        
    Returns:
        Tuple contenant:
        - Vecteurs des légendes
        - Longueur maximale des légendes
        - Objet tokenizer
    """
    print("Tokenisation des légendes...")
    tokenizer = tf.keras.preprocessing.text.Tokenizer(
        num_words=Config.VOCAB_SIZE,
        oov_token='<unk>',
        filters='!\"#$%&()*+.,-/:;=?@[\\]^_`{|}~ '
    )
    tokenizer.fit_on_texts(captions)
    
    # Assurer que <pad> est à l'index 0
    tokenizer.word_index['<pad>'] = 0
    tokenizer.index_word[0] = '<pad>'

    # Conversion des textes en séquences
    sequences = tokenizer.texts_to_sequences(captions)
    
    # Calcul de la longueur maximale des légendes
    max_length = max(len(caption.split()) for caption in captions)
    
    # Padding des séquences
    caption_vectors = tf.keras.preprocessing.sequence.pad_sequences(
        sequences, 
        padding='post', 
        maxlen=max_length
    )
    
    return caption_vectors, max_length, tokenizer

## Créer le CNN & RNN

La classe suivante définit un encodeur CNN, utilisé pour transformer les caractéristiques extraites par un modèle InceptionV3 en un espace d'embedding plus adapté à la génération de légendes. Le modèle comprend trois composants principaux :

- Dense Layer : Une couche dense qui réduit la dimension des caractéristiques extraites (en utilisant un embedding de taille embedding_dim).

- Batch Normalization : Une couche de normalisation qui ajuste les activations pour accélérer l'entraînement et améliorer la stabilité du modèle.

- Dropout : Une couche de régularisation qui aide à prévenir le sur-apprentissage en désactivant aléatoirement certaines connexions pendant l'entraînement.

Lors de l'entraînement, les caractéristiques de l'image passent à travers ces couches pour être transformées en un vecteur dense qui sera ensuite utilisé par les réseaux récurrents pour générer des légendes.


In [ ]:
class CNNEncoder(tf.keras.Model):
    """Encodeur CNN qui transforme les caractéristiques de l'image"""
    def __init__(self, embedding_dim: int):
        super(CNNEncoder, self).__init__(name="cnn_encoder")
        self.dense = tf.keras.layers.Dense(embedding_dim, name="encoder_dense")
        self.dropout = tf.keras.layers.Dropout(0.3, name="encoder_dropout")
        self.bn = tf.keras.layers.BatchNormalization(name="encoder_batch_norm")
        
    def call(self, features: tf.Tensor, training: bool = False) -> tf.Tensor:
        """
        Passe avant de l'encodeur
        
        Args:
            features: Caractéristiques de l'image extraites par InceptionV3
            training: Indique si le modèle est en mode entraînement
            
        Returns:
            Caractéristiques transformées
        """
        x = self.dense(features)
        x = self.bn(x, training=training)
        x = tf.nn.relu(x)
        if training:
            x = self.dropout(x)
        return x

La classe BahdanauAttention implémente un mécanisme d'attention permettant au modèle de se concentrer sur les parties les plus importantes de l'image à chaque étape de la génération de texte.

Elle utilise :

- Deux couches Dense (W1 et W2) pour calculer les scores d'attention.

- Une couche V pour générer un score final.

- Softmax pour normaliser les scores en poids d'attention.

Le résultat est un vecteur de contexte pondéré par les scores d'attention et les poids d'attention, permettant au modèle de générer des légendes plus précises.


In [ ]:
class BahdanauAttention(tf.keras.Model):
    """Mécanisme d'attention de Bahdanau"""
    def __init__(self, units: int):
        super(BahdanauAttention, self).__init__(name="bahdanau_attention")
        self.W1 = tf.keras.layers.Dense(units, name="attention_w1")
        self.W2 = tf.keras.layers.Dense(units, name="attention_w2")
        self.V = tf.keras.layers.Dense(1, name="attention_v")
    
    def call(self, features: tf.Tensor, hidden: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
        """
        Calcule les poids d'attention et le vecteur de contexte
        
        Args:
            features: Caractéristiques de l'image encodées
            hidden: État caché du décodeur RNN
            
        Returns:
            Tuple contenant:
            - Vecteur de contexte
            - Poids d'attention
        """
        # Expansion de la dimension temporelle pour l'état caché
        hidden_with_time_axis = tf.expand_dims(hidden, 1)
        
        # Calcul du score d'attention
        attention_hidden_layer = tf.nn.tanh(
            self.W1(features) + self.W2(hidden_with_time_axis)
        )
        score = self.V(attention_hidden_layer)
        
        # Normalisation des poids d'attention
        attention_weights = tf.nn.softmax(score, axis=1)
        
        # Calcul du vecteur de contexte
        context_vector = attention_weights * features
        context_vector = tf.reduce_sum(context_vector, axis=1)
        
        return context_vector, attention_weights


La classe RNNDecoder implémente un décodeur RNN avec un mécanisme d'attention pour la génération de texte. Elle comprend plusieurs composants essentiels :

- Embedding Layer : Convertit les indices des mots en vecteurs d'embedding.

- GRU Layer : Une couche GRU pour traiter les séquences et capturer les dépendances temporelles.

- Dense Layers : Deux couches denses pour transformer la sortie du GRU et prédire les mots.

- Dropout : Une couche de régularisation pour prévenir le sur-apprentissage.

- Attention Mechanism : Le mécanisme d'attention (Bahdanau) pour se concentrer sur les parties importantes des caractéristiques de l'image.

Lors du passage avant, les entrées passent d'abord par l'attention, puis les embeddings et le GRU génèrent les prédictions de mots et mettent à jour l'état caché. Le modèle retourne les prédictions, l'état caché et les poids d'attention.

In [ ]:
class RNNDecoder(tf.keras.Model):
    """Décodeur RNN avec mécanisme d'attention"""
    def __init__(self, embedding_dim: int, units: int, vocab_size: int):
        super(RNNDecoder, self).__init__(name="rnn_decoder")
        self.units = units
        
        # Couches du décodeur
        self.embedding = tf.keras.layers.Embedding(
            vocab_size, embedding_dim, name="decoder_embedding"
        )
        self.gru = tf.keras.layers.GRU(
            units,
            return_sequences=True,
            return_state=True,
            recurrent_initializer='glorot_uniform',
            recurrent_dropout=0.1,
            name="decoder_gru"
        )
        self.fc1 = tf.keras.layers.Dense(units, name="decoder_fc1")
        self.dropout = tf.keras.layers.Dropout(0.5, name="decoder_dropout")
        self.fc2 = tf.keras.layers.Dense(vocab_size, name="decoder_output")
        
        # Mécanisme d'attention
        self.attention = BahdanauAttention(units)

    def call(
        self, 
        x: tf.Tensor, 
        features: tf.Tensor, 
        hidden: tf.Tensor, 
        training: bool = False
    ) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
        """
        Passe avant du décodeur
        
        Args:
            x: Entrée du décodeur (tokens embeddings)
            features: Caractéristiques de l'image encodées
            hidden: État caché précédent
            training: Indique si le modèle est en mode entraînement
            
        Returns:
            Tuple contenant:
            - Prédictions de mots
            - Nouvel état caché
            - Poids d'attention
        """
        # Calcul du vecteur de contexte avec attention
        context_vector, attention_weights = self.attention(features, hidden)
        
        # Conversion des indices en embeddings
        x = self.embedding(x)
        
        # Concaténation du vecteur de contexte et des embeddings
        context_vector_expanded = tf.expand_dims(context_vector, 1)
        x = tf.concat([context_vector_expanded, x], axis=-1)
        
        # Passage par le GRU
        output, state = self.gru(x)
        
        # Reshape de la sortie
        x = self.fc1(output)
        if training:
            x = self.dropout(x)
        x = tf.reshape(x, (-1, x.shape[2]))
        
        # Prédiction du vocabulaire
        x = self.fc2(x)
        
        return x, state, attention_weights

    def initialize_hidden_state(self, batch_size: int) -> tf.Tensor:
        """
        Initialise l'état caché du décodeur
        
        Args:
            batch_size: Taille du batch
            
        Returns:
            État caché initial (zéros)
        """
        return tf.zeros((batch_size, self.units))


## Extraction des caractéristiques de l'image

Cette fonction utilise InceptionV3 (pré-entraîné sur ImageNet) pour extraire des caractéristiques visuelles à partir d’un dataset d’images. Ces caractéristiques, représentant des vecteurs abstraits des images, sont ensuite sauvegardées individuellement au format .npy pour un usage ultérieur, comme l’entraînement d’un modèle de génération de légendes. Le modèle retourné est celui utilisé pour cette extraction.

In [ ]:
def extract_image_features(image_dataset: tf.data.Dataset) -> tf.keras.Model:
    """
    Extrait et sauvegarde les caractéristiques des images avec InceptionV3
    
    Args:
        image_dataset: Dataset contenant les images à traiter
        
    Returns:
        Modèle d'extraction de caractéristiques
    """
    print("Chargement d'InceptionV3...")
    image_model = tf.keras.applications.InceptionV3(include_top=False, weights='imagenet')
    
    print("Création du modèle d'extraction...")
    image_features_extract_model = tf.keras.Model(
        inputs=image_model.input, 
        outputs=image_model.layers[-1].output
    )
    
    print("Extraction des caractéristiques des images...")
    for img, path in tqdm(image_dataset):
        batch_features = image_features_extract_model(img)
        batch_features = tf.reshape(
            batch_features, 
            (batch_features.shape[0], -1, batch_features.shape[3])
        )
        
        for bf, p in zip(batch_features, path):
            path_of_feature = p.numpy().decode('utf-8')
            np.save(path_of_feature, bf.numpy())
    
    return image_features_extract_model

## Création du dataset

On prépare les datasets TensorFlow à partir des images et de leurs légendes vectorisées.
Elle associe chaque image à ses légendes, répartit les données entre entraînement et validation, charge les caractéristiques extraites (.npy), ajuste les séquences pour le décodeur, et applique batching, shuffling et préchargement pour une efficacité maximale. Elle retourne les deux jeux de données, avec les tailles correspondantes.

In [ ]:
def create_train_val_datasets(
    img_name_vector: List[str], 
    caption_vectors: np.ndarray, 
    tokenizer: Any, 
    max_length: int
) -> Tuple[tf.data.Dataset, tf.data.Dataset, int, int]:
    """
    Crée les datasets d'entraînement et de validation
    
    Args:
        img_name_vector: Liste des chemins d'images
        caption_vectors: Vecteurs des légendes tokenisées
        tokenizer: Tokenizer utilisé
        max_length: Longueur maximale des légendes
        
    Returns:
        Tuple contenant:
        - Dataset d'entraînement
        - Dataset de validation
        - Nombre de légendes d'entraînement
        - Nombre de légendes de validation
    """
    print("Préparation des jeux d'entraînement et de validation...")
    
    # Création du dictionnaire associant chaque image à ses légendes vectorisées
    image_to_caption_vector = collections.defaultdict(list)
    for img, cap in zip(img_name_vector, caption_vectors):
        image_to_caption_vector[img].append(cap)
    
    # Division du dataset
    image_keys = list(image_to_caption_vector.keys())
    random.shuffle(image_keys)
    train_size = int(len(image_keys) * (1 - Config.VALIDATION_SPLIT))
    train_image_keys = image_keys[:train_size]
    val_image_keys = image_keys[train_size:]
    
    # Extraction des légendes et chemins d'images pour chaque ensemble
    def extract_image_and_caption(keys, image_to_caption_dict):
        caps = []
        img_names = []
        for img in keys:
            caption_list = image_to_caption_dict[img]
            caps.extend(caption_list)
            img_names.extend([img] * len(caption_list))
        return caps, img_names
    
    train_captions, train_img_names = extract_image_and_caption(train_image_keys, image_to_caption_vector)
    val_captions, val_img_names = extract_image_and_caption(val_image_keys, image_to_caption_vector)
    
    # Création des datasets TensorFlow
    def map_func(image_name, caption):
        img_tensor = np.load(image_name.decode('utf-8') + '.npy')
        return img_tensor, caption
    
    def create_dataset(img_names, captions):
        dataset = tf.data.Dataset.from_tensor_slices((img_names, captions))
        dataset = dataset.map(
            lambda img, cap: tf.numpy_function(
                map_func, [img, cap], [tf.float32, tf.int32]
            ),
            num_parallel_calls=tf.data.AUTOTUNE
        )
        
        # Définition explicite des formes des tenseurs
        dataset = dataset.map(
            lambda img_tensor, cap: (
                tf.ensure_shape(img_tensor, [Config.ATTENTION_FEATURES_SHAPE, Config.FEATURES_SHAPE]),
                tf.ensure_shape(cap, [max_length])
            )
        )
        
        # Création des entrées et cibles du décodeur
        dataset = dataset.map(
            lambda img_tensor, cap: (img_tensor, cap[:-1], cap[1:])
        )
        
        # Batch et shuffle
        dataset = dataset.shuffle(Config.BUFFER_SIZE).batch(Config.BATCH_SIZE, drop_remainder=True)
        dataset = dataset.prefetch(buffer_size=tf.data.AUTOTUNE)
        
        return dataset
    
    print(f"Création de {len(train_captions)} exemples d'entraînement et {len(val_captions)} exemples de validation...")
    train_dataset = create_dataset(train_img_names, train_captions)
    val_dataset = create_dataset(val_img_names, val_captions)
    
    return train_dataset, val_dataset, len(train_captions), len(val_captions)

## Fonction de perte

On calcule la perte de type cross-entropie entre les prédictions et les vraies étiquettes,
tout en ignorant les tokens de padding (<pad>) via un masque.
Elle retourne la perte moyenne sur les éléments valides de la séquence.

In [ ]:
def loss_function(real: tf.Tensor, pred: tf.Tensor) -> tf.Tensor:
    """
    Calcule la perte avec masquage pour ignorer le padding
    
    Args:
        real: Étiquettes réelles
        pred: Prédictions du modèle
        
    Returns:
        Valeur de perte
    """
    loss_object = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction='none')
    
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_object(real, pred)

    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask

    return tf.reduce_mean(loss_)

## Fonctions d'entraînement et de validation

In [ ]:
@tf.function
def train_step(
    encoder: CNNEncoder, 
    decoder: RNNDecoder, 
    optimizer: tf.keras.optimizers.Optimizer,
    tokenizer: Any,
    img_tensor: tf.Tensor, 
    input_seq: tf.Tensor, 
    target_seq: tf.Tensor
) -> Tuple[tf.Tensor, tf.Tensor]:
    """
    Effectue une étape d'entraînement
    
    Args:
        encoder: Modèle encodeur
        decoder: Modèle décodeur
        optimizer: Optimiseur
        tokenizer: Tokenizer utilisé
        img_tensor: Caractéristiques de l'image
        input_seq: Séquence d'entrée du décodeur
        target_seq: Séquence cible du décodeur
        
    Returns:
        Tuple contenant:
        - Perte totale
        - Perte moyenne
    """
    loss = tf.constant(0.0, dtype=tf.float32)
    
    with tf.GradientTape() as tape:
        # Encoder l'image
        features = encoder(img_tensor, training=True)
        
        # Initialiser l'état caché du décodeur
        hidden = decoder.initialize_hidden_state(batch_size=target_seq.shape[0])
        
        # Préparer l'entrée du décodeur pour l'étape courante
        dec_input = tf.expand_dims([tokenizer.word_index['<start>']] * target_seq.shape[0], 1)
            
        # Traitement token par token
        for i in range(input_seq.shape[1]):
            # Obtenir la sortie du décodeur
            predictions, hidden, _ = decoder(dec_input, features, hidden, training=True)
            
            # Calculer la perte pour le token courant
            step_loss = loss_function(target_seq[:, i], predictions)
            loss += step_loss

            dec_input = tf.expand_dims(input_seq[:, i], 1)
            
    # Perte moyenne
    total_loss = loss / int(target_seq.shape[1])
    
    # Calcul des gradients
    trainable_variables = encoder.trainable_variables + decoder.trainable_variables
    gradients = tape.gradient(loss, trainable_variables)
    print(f"DEBUG: Gradients = {[grad if grad is not None else 'None' for grad in gradients]}")
    
    print(f"DEBUG: Loss = {loss}, Total Loss = {total_loss}")
    # Vérifier si des gradients sont calculés
    if not any(grad is not None for grad in gradients):
        print("DEBUG: Gradients non calculés. Vérifiez les entrées, sorties et la perte.")
        print(f"DEBUG: img_tensor.shape = {img_tensor.shape}")
        print(f"DEBUG: input_seq.shape = {input_seq.shape}")
        print(f"DEBUG: target_seq.shape = {target_seq.shape}")
        raise ValueError("No gradients provided for any variable. Check the model and loss computation.")

    # Application des gradients
    optimizer.apply_gradients(zip(gradients, trainable_variables))
    
    return loss, total_loss

## Description de la fonction `validation_step()`

La fonction `validation_step()` effectue une étape de validation pour évaluer la performance du modèle sur un lot de données, sans mise à jour des poids.

### Objectif
Calculer la perte (loss) entre les prédictions du modèle et les vraies légendes pendant la phase de validation.

### Paramètres

- `encoder` : modèle CNN qui encode les caractéristiques visuelles.
- `decoder` : modèle RNN qui génère la légende.
- `img_tensor` : tenseur contenant les caractéristiques extraites de l’image.
- `input_seq` : séquence d’entrée (tokens de la légende sans le dernier mot).
- `target_seq` : séquence cible (tokens de la légende sans le premier mot).

### Étapes

1. **Encodage** de l’image avec l’`encoder` (mode validation).
2. **Initialisation** de l’état caché du `decoder`.
3. **Boucle de génération mot par mot** :
   - À chaque étape, le `decoder` prédit un mot.
   - La perte est calculée entre la prédiction et le mot attendu.
   - La perte est accumulée.
4. **Retour de la perte moyenne** par mot.


In [ ]:
@tf.function
def validation_step(
    encoder: CNNEncoder, 
    decoder: RNNDecoder, 
    img_tensor: tf.Tensor, 
    input_seq: tf.Tensor, 
    target_seq: tf.Tensor
) -> tf.Tensor:
    """
    Effectue une étape de validation
    
    Args:
        encoder: Modèle encodeur
        decoder: Modèle décodeur
        img_tensor: Caractéristiques de l'image
        input_seq: Séquence d'entrée du décodeur
        target_seq: Séquence cible du décodeur
        
    Returns:
        Perte de validation
    """
    loss = 0
    
    # Encoder l'image (pas de mode training)
    features = encoder(img_tensor, training=False)
    
    # Initialiser l'état caché du décodeur
    hidden = decoder.initialize_hidden_state(batch_size=input_seq.shape[0])
    
    # Traitement token par token
    for i in range(input_seq.shape[1]):
        # Préparer l'entrée du décodeur pour l'étape courante
        dec_input = tf.expand_dims(input_seq[:, i], 1)
        
        # Obtenir la sortie du décodeur (pas de mode training)
        predictions, hidden, _ = decoder(dec_input, features, hidden, training=False)
        
        # Calculer la perte pour le token courant
        step_loss = loss_function(target_seq[:, i], predictions)
        loss += step_loss
    
    # Perte moyenne
    return loss / int(target_seq.shape[1])


### Résultat

Retourne la **perte moyenne** pour la séquence, utilisée pour évaluer la qualité de la génération sans modifier le modèle.

## Evaluation du modèle

## Description de la fonction `evaluate_model()`

La fonction `evaluate_model()` génère une légende automatique pour une image en utilisant un modèle Encoder-Decoder avec attention.

### Objectif
Décrire une image en texte à partir d’un modèle entraîné.

### Paramètres

- `encoder` : modèle qui encode les caractéristiques de l’image.
- `decoder` : modèle qui génère les mots de la légende.
- `image_features_extract_model` : modèle CNN pour extraire les caractéristiques de l’image.
- `tokenizer` : convertit les mots en indices et inversement.
- `max_length` : longueur maximale de la légende.
- `image_path` : chemin de l’image à décrire.

### Étapes

1. **Préparation de l’image** : chargement et extraction des caractéristiques.
2. **Encodage** : transformation des caractéristiques en vecteurs utilisables par le décodeur.
3. **Début de génération** : le décodeur commence avec le token `<start>`.
4. **Boucle de prédiction** :
   - À chaque étape, un mot est prédit.
   - Les poids d’attention sont enregistrés.
   - La boucle s’arrête si le mot `<end>` est généré ou si la longueur maximale est atteinte.
5. **Retour** :
   - Liste des mots générés.
   - Matrice des poids d’attention.

In [1]:
def evaluate_model(
    encoder: CNNEncoder, 
    decoder: RNNDecoder, 
    image_features_extract_model: tf.keras.Model,
    tokenizer: Any, 
    max_length: int, 
    image_path: str
) -> Tuple[List[str], np.ndarray]:
    """
    Génère une légende pour une image
    
    Args:
        encoder: Modèle encodeur
        decoder: Modèle décodeur
        image_features_extract_model: Modèle d'extraction de caractéristiques
        tokenizer: Tokenizer pour le texte
        max_length: Longueur maximale des légendes
        image_path: Chemin de l'image à évaluer
        
    Returns:
        Tuple contenant:
        - Liste des mots prédits
        - Matrice des poids d'attention
    """
    attention_plot = np.zeros((max_length, Config.ATTENTION_FEATURES_SHAPE))

    # Chargement et prétraitement de l'image
    temp_image, _ = load_and_preprocess_image(image_path)
    img_tensor = tf.expand_dims(temp_image, 0)
    
    # Extraction des caractéristiques
    img_tensor_features = image_features_extract_model(img_tensor)
    img_tensor_features = tf.reshape(
        img_tensor_features, 
        (img_tensor_features.shape[0], -1, img_tensor_features.shape[3])
    )
    
    # Encodage des caractéristiques
    features = encoder(img_tensor_features, training=False)

    # Initialisation du décodeur
    hidden = decoder.initialize_hidden_state(batch_size=1)
    dec_input = tf.expand_dims([tokenizer.word_index['<start>']], 0)
    
    result = []
    
    # Génération mot par mot
    for i in range(max_length):
        predictions, hidden, attention_weights = decoder(
            dec_input, features, hidden, training=False
        )
        
        # Stockage des poids d'attention
        attention_plot[i] = tf.reshape(attention_weights, (-1,)).numpy()
        
        # Prédiction du prochain mot
        predicted_id = tf.random.categorical(predictions, 1)[0][0].numpy()
        result.append(tokenizer.index_word[predicted_id])
        
        # Arrêt si "<end>" est prédit
        if tokenizer.index_word[predicted_id] == '<end>':
            break
        
        # Préparation de l'entrée pour la prochaine itération
        dec_input = tf.expand_dims([predicted_id], 0)
    
    attention_plot = attention_plot[:len(result), :]
    return result, attention_plot

NameError: name 'CNNEncoder' is not defined

### Résultat

La fonction retourne :
- Une légende prédite (sous forme de liste de mots).
- Les poids d’attention pour chaque mot généré (pour visualisation).

## Entrainement du modèle

## Description de la fonction `train_model()`

La fonction `train_model()` constitue le pipeline complet d'entraînement d’un modèle de génération de légendes pour des images. Elle suit une architecture Encoder-Decoder avec attention, utilisant TensorFlow. Voici un aperçu structuré des étapes du processus :

### 1. Initialisation
- **Configuration des répertoires** de travail nécessaires à l’entraînement.
- **Chargement des annotations** contenant les couples (image, légende).

### 2. Préparation des données
- Création d’un dataset TensorFlow à partir des chemins d’images.
- Application du prétraitement sur les images : redimensionnement, normalisation, etc.
- Découpage en batchs pour l’entraînement.

### 3. Extraction des caractéristiques
- Utilisation d’un modèle de type CNN (pré-entraîné) pour extraire des vecteurs de caractéristiques des images.

### 4. Prétraitement des légendes
- **Tokenisation** des légendes : transformation en séquences numériques.
- Calcul de la longueur maximale des séquences.

### 5. Création des datasets d'entraînement et de validation
- Division des données en deux ensembles : entraînement et validation.
- Mise en forme sous forme de tuples `(image_tensor, input_sequence, target_sequence)`.

### 6. Construction du modèle
- Initialisation de l’encodeur (CNNEncoder) et du décodeur (RNNDecoder).
- Définition de l’optimiseur (Adam) et du système de gestion des checkpoints.

### 7. Boucle d'entraînement
- Exécution d’une boucle sur le nombre d’époques défini.
- Calcul et suivi des pertes d'entraînement et de validation.
- Sauvegarde du meilleur modèle (basé sur la perte de validation).
- Enregistrement régulier des checkpoints.

### 8. Visualisation des performances
- Génération et sauvegarde de graphiques illustrant l’évolution des pertes au cours de l’entraînement.

### 9. Sauvegarde finale
- Sauvegarde des versions finales de l’encodeur, du décodeur, et du tokenizer.
- Export des métadonnées du modèle (hyperparamètres, dimensions, date d'entraînement, etc.).

### 10. Évaluation qualitative
- Sélection d’un échantillon d’images du jeu de validation.
- Génération de légendes prédites et comparaison avec les légendes réelles.
- Visualisation des images et des cartes d’attention associées.

In [ ]:
def train_model() -> None:
    """Fonction principale pour l'entraînement du modèle"""
    # Configuration des répertoires
    setup_directories()

    # 1. Chargement des annotations
    captions, img_name_vector, image_path_to_caption = load_annotations()

    # 2. Création du dataset d'images
    print("Création du dataset d'images...")
    img_names = sorted(set(img_name_vector))
    image_dataset = tf.data.Dataset.from_tensor_slices(img_names)
    image_dataset = image_dataset.map(
        load_and_preprocess_image, 
        num_parallel_calls=tf.data.AUTOTUNE
    ).batch(Config.BATCH_SIZE)

    # 3. Extraction des caractéristiques des images
    image_features_extract_model = extract_image_features(image_dataset)

    # 4. Tokenisation des légendes
    caption_vectors, max_length, tokenizer = build_tokenizer(captions)

    # 5. Création des datasets d'entraînement et de validation
    train_dataset, val_dataset, train_size, val_size = create_train_val_datasets(
        img_name_vector, caption_vectors, tokenizer, max_length
    )

    # 6. Création du modèle
    print("Création du modèle...")
    encoder = CNNEncoder(Config.EMBEDDING_DIM)
    decoder = RNNDecoder(Config.EMBEDDING_DIM, Config.UNITS, len(tokenizer.word_index) + 1)

    # 7. Création de l'optimiseur et du checkpoint
    optimizer = tf.keras.optimizers.Adam(learning_rate=Config.LEARNING_RATE)
    checkpoint = tf.train.Checkpoint(
        encoder=encoder,
        decoder=decoder,
        optimizer=optimizer
    )
    checkpoint_manager = tf.train.CheckpointManager(
        checkpoint,
        Config.CHECKPOINT_PATH,
        max_to_keep=5
    )
    # 8. Entraînement du modèle
    print("Début de l'entraînement...")
    start_epoch = 0
    if checkpoint_manager.latest_checkpoint:
        start_epoch = int(checkpoint_manager.latest_checkpoint.split('-')[-1])
        checkpoint.restore(checkpoint_manager.latest_checkpoint)
        print(f"Restauration depuis {checkpoint_manager.latest_checkpoint}")
    
    # Variables pour suivre les métriques
    train_loss_history = []
    val_loss_history = []
    best_val_loss = float('inf')
    steps_per_epoch = train_size // Config.BATCH_SIZE
    
    for epoch in range(start_epoch, Config.EPOCHS):
        start = time.time()
        total_train_loss = 0
        total_val_loss = 0
        
        # Entraînement sur une époque
        for (batch, (img_tensor, inp, target)) in enumerate(train_dataset):
            batch_loss, t_loss = train_step(encoder, decoder, optimizer, 
                                           tokenizer, img_tensor, inp, target)
            total_train_loss += t_loss
            
            if batch % 100 == 0:
                print(f"Epoch {epoch+1}/{Config.EPOCHS} - "
                      f"Batch {batch}/{steps_per_epoch} - "
                      f"Loss {batch_loss.numpy()/target.shape[1]:.4f}")
        
        # Calcul de la perte moyenne sur l'époque
        avg_train_loss = total_train_loss / steps_per_epoch
        train_loss_history.append(avg_train_loss.numpy())
        
        # Validation
        for img_tensor, inp, target in val_dataset:
            batch_val_loss = validation_step(encoder, decoder, img_tensor, inp, target)
            total_val_loss += batch_val_loss
        
        avg_val_loss = total_val_loss / (val_size // Config.BATCH_SIZE)
        val_loss_history.append(avg_val_loss.numpy())
        
        # Affichage des résultats de l'époque
        print(f'Epoch {epoch+1}/{Config.EPOCHS}:')
        print(f'  - Train Loss: {avg_train_loss:.6f}')
        print(f'  - Validation Loss: {avg_val_loss:.6f}')
        print(f'  - Temps: {time.time()-start:.2f} sec')
        
        # Sauvegarde du meilleur modèle
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            # Sauvegarde spéciale pour le meilleur modèle
            encoder_path = os.path.join(Config.MODEL_SAVE_PATH, 'best_encoder.keras')
            decoder_path = os.path.join(Config.MODEL_SAVE_PATH, 'best_decoder.keras')
            encoder.save(encoder_path)
            decoder.save(decoder_path)
            print(f"  - Nouveau meilleur modèle sauvegardé (val_loss: {best_val_loss:.6f})")
        
        # Sauvegarde régulière du checkpoint
        if (epoch + 1) % 5 == 0 or epoch == Config.EPOCHS - 1:
            checkpoint_manager.save()
    
    # 9. Tracé des courbes d'apprentissage
    plt.figure(figsize=(12, 6))
    plt.subplot(1, 2, 1)
    plt.plot(train_loss_history, label='Train Loss')
    plt.plot(val_loss_history, label='Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Loss Evolution')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(train_loss_history, label='Train Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.title('Training Loss Detail')
    plt.savefig('training_history.png')
    plt.close()
    
    # 10. Sauvegarde des modèles et configurations finaux
    print("Sauvegarde des modèles et configurations...")
    
    # Sauvegarde des modèles finaux
    encoder_path = os.path.join(Config.MODEL_SAVE_PATH, 'final_encoder.keras')
    decoder_path = os.path.join(Config.MODEL_SAVE_PATH, 'final_decoder.keras')
    encoder.save(encoder_path)
    decoder.save(decoder_path)
    
    # Sauvegarde du tokenizer
    tokenizer_path = os.path.join(Config.MODEL_SAVE_PATH, 'tokenizer.pickle')
    with open(tokenizer_path, 'wb') as handle:
        pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
    # Sauvegarde des métadonnées du modèle
    metadata = {
        'max_length': max_length,
        'embedding_dim': Config.EMBEDDING_DIM,
        'units': Config.UNITS,
        'vocab_size': Config.VOCAB_SIZE,
        'features_shape': Config.FEATURES_SHAPE,
        'attention_features_shape': Config.ATTENTION_FEATURES_SHAPE,
        'img_size': Config.IMG_SIZE,
        'training_date': time.strftime('%Y-%m-%d %H:%M:%S')
    }
    
    metadata_path = os.path.join(Config.MODEL_SAVE_PATH, 'model_metadata.json')
    with open(metadata_path, 'w') as f:
        json.dump(metadata, f)
    
    # 11. Évaluation sur quelques images de validation
    print("\nÉvaluation du modèle sur quelques exemples...")
    
    # Sélectionner quelques images de validation aléatoirement
    val_image_paths = random.sample(img_names, 5)
    
    plt.figure(figsize=(20, 20))
    
    for i, image_path in enumerate(val_image_paths):
        # Génération de la légende
        result, attention_plot = evaluate_model(
            encoder, 
            decoder, 
            image_features_extract_model,
            tokenizer,
            max_length,
            image_path
        )
        
        # Récupération de la légende réelle
        real_captions = image_path_to_caption[image_path]
        real_caption = real_captions[0].replace('<start> ', '').replace(' <end>', '')
        
        # Affichage des résultats
        predicted_caption = ' '.join([word for word in result if word not in ['<start>', '<end>']])
        
        print(f"\nImage {i+1}:")
        print(f"  - Légende réelle: {real_caption}")
        print(f"  - Légende prédite: {predicted_caption}")
        
        # Visualisation de l'image et de l'attention
        plt.subplot(5, 2, 2*i+1)
        temp_image = plt.imread(image_path)
        plt.imshow(temp_image)
        plt.title(f"Image {i+1}")
        plt.axis('off')
        
        plt.subplot(5, 2, 2*i+2)
        attention_mean = np.mean(attention_plot, axis=0)
        attention_reshaped = np.reshape(attention_mean, (8, 8))
        plt.imshow(attention_reshaped)
        plt.title("Carte d'attention moyenne")
        
    plt.savefig('evaluation_examples.png')
    plt.close()
    
    print("\nEntraînement terminé avec succès!")
    print(f"Modèles sauvegardés dans: {Config.MODEL_SAVE_PATH}")

## Résultat
Un modèle complet et entraîné est généré, prêt à être utilisé pour la génération automatique de descriptions d’images, avec un historique de performance et tous les artefacts nécessaires à la reproduction ou au déploiement du modèle.

In [ ]:
train_model()